In [ ]:
import numpy as np
from netgen.csg import *
from netgen.meshing import MeshingStep
from ngsolve import *
from ngsolve.webgui import Draw
ngsglobals.msg_level = 1

# Generate mesh and plot it
center = Pnt(0,0,0)
R = 1.0
geo = CSGeometry()
geo.Add(Sphere(center, R))
maxh = 1.0
mesh = Mesh(geo.GenerateMesh(maxh=maxh, perfstepsend=MeshingStep.MESHSURFACE))
#   We use a superparametric approximation to ensure 
#   optimal convergence of solution also in w.r.t to normal component 
order_k = 3
order_g = order_k+1
mesh.Curve(order_g)
Draw(mesh)

# Check convergence of normal vector, mean curvature  (and B itself)
Hhb_errs = []
H_errs = []

def compute_eoc(errs):
    return np.log(np.array(errs)[:-1]/np.array(errs)[1:])/np.log(2)

In [ ]:
# Define geometry related objects and differential operators

# Mesh size
h = specialcf.mesh_size

# Normal vector
ns = specialcf.normal(mesh.dim)
# normal and tangential  projection
Qs = OuterProduct(ns, ns)
Ps = Id(mesh.dim) - Qs
# Weingarten map
Bs = Grad(ns)
# TODO: Or should we rather use
# specialcf.Weingarten(mesh.dim)

# Define covariant derivative for vectors
def Nabla_h(u):
    return Ps*Grad(u).Trace()*Ps
# and its corrected version for non-tangential vectors
def Nabla_th(u):
    return Nabla(u) - InnerProduct(u, ns)*Bs

# Define discrete strain rate tensor
# TODO: Need to clarify meaning/implementation of Grad(u).Trace()
#       for VectorH1
# TODO: Which projection Ps do we actually need
def E_h(u):
    return 0.5*Ps*Sym(Grad(u).Trace())*Ps
    # return 0.5*Ps*(Grad(u).Trace())
    # return Ps*(grad(u).Trace())

# Corrected strain rate tensor for non-tangential 
def E_th(u):
    return E_h(u) - InnerProduct(u, ns)*Bs

V = VectorH1(mesh, order=int(order_k))
# V = HDivSurface(mesh, order=int(order_k))
# V = H1(mesh, order=int(order_k))
u, v = V.TnT()

# Penalty parameter to weakly enforce tangential condition
beta = 1.0

Astar = BilinearForm(V)
# Strain rates 
Astar += 2*InnerProduct(E_th(u), E_th(v))*ds
# Mass terms to ensure trivial kernel
# TODO: DO we need Traces here?
Astar += InnerProduct(Ps*u.Trace(), Ps*v.Trace())*ds
# TODO: Last term can replaced with
# Astar += InnerProduct(u.Trace(), v.Trace())*ds
# Penalization of normal component
# TODO:More efficient way would be to compute (u,n)*(v*n) directly
# Test that!
Astar += beta*h**(-2.0)*InnerProduct(Qs*u.Trace(), Qs*v.Trace())*ds
Astar.Assemble()

# Tangential test field
uex = CF((-y*(1-z)*(1+z), x*(1-z)*(1+z), 0))
uex_h = GridFunction(V)
uex_h.Set(uex, definedon=mesh.Boundaries(".*"))
l = LinearForm(V)
l += 2*InnerProduct(E_th(uex_h), E_th(v))*ds
l += InnerProduct(Ps*uex_h.Trace(), Ps*v.Trace())*ds
l.Assemble()

In [ ]:
# Solve problem and write out solution
u_h =  GridFunction(V)
Ainv = Astar.mat.Inverse(freedofs=V.FreeDofs())
u_h.vec.data = Ainv*l.vec

Draw(u_h)
vtkfile = VTKOutput(mesh, coefs=[u_h, uex], names=["u_h", "uex"], filename="results",subdivision=3)
vtkfile.Do(vb=BND)
